<a href="https://colab.research.google.com/github/gloryyy17/AI-Excuse-Generator/blob/main/cm47_mv_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

In [ ]:
!mkdir -p ~/.kaggle
!echo "KGAT_0c7cd0300c477bb68a6936ae386cbc33" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

In [ ]:
!pip install -q kaggle

In [ ]:
!kaggle datasets download -d dataclusterlabs/indian-sign-board-image-dataset

Dataset URL: https://www.kaggle.com/datasets/dataclusterlabs/indian-sign-board-image-dataset
License(s): copyright-authors
100% 400M/400M [00:19<00:00, 21.9MB/s]



In [ ]:
!unzip -q indian-sign-board-image-dataset.zip -d dataset

In [ ]:
import os

for root, dirs, files in os.walk("/content/dataset"):
    print(root)
    print("Folders:", dirs)
    print("Files:", files[:5])
    print("-"*50)

/content/dataset
Folders: ['Annotations', 'Batch-1']
Files: []
--------------------------------------------------
/content/dataset/Annotations
Folders: ['Annotations']
Files: []
--------------------------------------------------
/content/dataset/Annotations/Annotations
Folders: []
Files: ['Datacluster Traffic Sign (13).xml', 'Datacluster Traffic Sign (89).xml', 'Datacluster Traffic Sign (53).xml', 'Datacluster Traffic Sign (142).xml', 'Datacluster Traffic Sign (81).xml']
--------------------------------------------------
/content/dataset/Batch-1
Folders: ['batch']
Files: []
--------------------------------------------------
/content/dataset/Batch-1/batch
Folders: []
Files: ['Datacluster Traffic Sign (115).jpg', 'Datacluster Traffic Sign (69).jpg', 'Datacluster Traffic Sign (11).jpg', 'Datacluster Traffic Sign (17).jpg', 'Datacluster Traffic Sign (18).jpg']
--------------------------------------------------


In [ ]:
orb = cv2.ORB_create(
    nfeatures=1000
)
def extract_features(image_path):

    img = cv2.imread(image_path)

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )
    # FAST Corner Detection
    fast = cv2.FastFeatureDetector_create()
    keypoints = fast.detect(gray,None)
    # ORB Feature Extraction
    keypoints, descriptors = orb.compute(
        gray,
        keypoints
    )
    return img, keypoints, descriptors

In [ ]:
!ls

dataset  indian-sign-board-image-dataset.zip  sample_data


In [ ]:
!pwd

/content


In [ ]:
!ls -lh

total 401M
drwxr-xr-x 4 root root 4.0K Jul  7 06:34 dataset
-rw-r--r-- 1 root root 401M Apr 22  2023 indian-sign-board-image-dataset.zip
drwxr-xr-x 1 root root 4.0K Jun  4 13:32 sample_data


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'dataset', 'indian-sign-board-image-dataset.zip', 'sample_data']


In [ ]:
folder = os.listdir("/content/dataset")[0]
print(folder)
print(os.listdir("/content/dataset/" + folder))

Annotations
['Annotations']


In [ ]:
import os

for root, dirs, files in os.walk("/content/dataset"):
    print("Folder:", root)
    print("Subfolders:", dirs)
    print("Files:", files[:5])   # First 5 files only
    print("-"*60)

Folder: /content/dataset
Subfolders: ['Annotations', 'Batch-1']
Files: []
------------------------------------------------------------
Folder: /content/dataset/Annotations
Subfolders: ['Annotations']
Files: []
------------------------------------------------------------
Folder: /content/dataset/Annotations/Annotations
Subfolders: []
Files: ['Datacluster Traffic Sign (13).xml', 'Datacluster Traffic Sign (89).xml', 'Datacluster Traffic Sign (53).xml', 'Datacluster Traffic Sign (142).xml', 'Datacluster Traffic Sign (81).xml']
------------------------------------------------------------
Folder: /content/dataset/Batch-1
Subfolders: ['batch']
Files: []
------------------------------------------------------------
Folder: /content/dataset/Batch-1/batch
Subfolders: []
Files: ['Datacluster Traffic Sign (115).jpg', 'Datacluster Traffic Sign (69).jpg', 'Datacluster Traffic Sign (11).jpg', 'Datacluster Traffic Sign (17).jpg', 'Datacluster Traffic Sign (18).jpg']
------------------------------------

In [ ]:
import os
import cv2
import xml.etree.ElementTree as ET

image_folder = "/content/dataset/Batch-1/batch"
annotation_folder = "/content/dataset/Annotations/Annotations"

crop_folder = "/content/cropped_signs"

os.makedirs(crop_folder, exist_ok=True)

count = 0

for xml_file in os.listdir(annotation_folder):

    if not xml_file.endswith(".xml"):
        continue

    tree = ET.parse(os.path.join(annotation_folder, xml_file))
    root = tree.getroot()

    filename = root.find("filename").text

    image_path = os.path.join(image_folder, filename)

    image = cv2.imread(image_path)

    if image is None:
        continue

    for obj in root.findall("object"):

        label = obj.find("name").text

        bbox = obj.find("bndbox")

        xmin = int(float(bbox.find("xmin").text))
        ymin = int(float(bbox.find("ymin").text))
        xmax = int(float(bbox.find("xmax").text))
        ymax = int(float(bbox.find("ymax").text))

        crop = image[ymin:ymax, xmin:xmax]

        save_name = f"{label}_{count}.jpg"

        cv2.imwrite(
            os.path.join(crop_folder, save_name),
            crop
        )

        count += 1

print("Total Cropped Images:", count)

Total Cropped Images: 194


In [ ]:
image_folder = "/content/dataset/Batch-1/batch"
annotation_folder = "/content/dataset/Annotations/Annotations"

In [ ]:
def get_bbox(xml_file):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    bbox = root.find("object").find("bndbox")

    xmin = int(bbox.find("xmin").text)
    ymin = int(bbox.find("ymin").text)
    xmax = int(bbox.find("xmax").text)
    ymax = int(bbox.find("ymax").text)

    return xmin,ymin,xmax,ymax


reference_images = {}


xml_files = os.listdir(annotation_folder)


for xml in xml_files:

    if xml.endswith(".xml"):

        image_name = xml.replace(".xml",".jpg")

        image_path = os.path.join(
            image_folder,
            image_name
        )


        if os.path.exists(image_path):

            img = cv2.imread(image_path)

            xmin,ymin,xmax,ymax = get_bbox(
                os.path.join(annotation_folder,xml)
            )


            sign = img[
                ymin:ymax,
                xmin:xmax
            ]


            sign_gray=cv2.cvtColor(
                sign,
                cv2.COLOR_BGR2GRAY
            )


            reference_images[image_name]=sign_gray



print(
    "Total reference signs:",
    len(reference_images)
)

AttributeError: 'NoneType' object has no attribute 'find'

In [ ]:
reference_images = {}


xml_files = os.listdir(annotation_folder)


for xml in xml_files:

    if xml.endswith(".xml"):

        image_name = xml.replace(".xml",".jpg")

        image_path = os.path.join(
            image_folder,
            image_name
        )


        if os.path.exists(image_path):

            img = cv2.imread(image_path)

            xmin,ymin,xmax,ymax = get_bbox(
                os.path.join(annotation_folder,xml)
            )


            sign = img[
                ymin:ymax,
                xmin:xmax
            ]


            sign_gray=cv2.cvtColor(
                sign,
                cv2.COLOR_BGR2GRAY
            )


            reference_images[image_name]=sign_gray



print(
    "Total reference signs:",
    len(reference_images)
)

AttributeError: 'NoneType' object has no attribute 'find'